# 02 · Build a neuron

## What you will build

A single artificial neuron, typed by you: the weighted sum, two activation functions, and a version that handles a whole batch at once. You will then set its weights by hand so it computes logic gates, draw the line it uses to decide, and find the one simple problem a single neuron cannot solve, which is why networks have layers.

## Learning objectives

- Write the neuron's two steps, $z = w \cdot x + b$ and $a = f(z)$, in NumPy.
- Implement the sigmoid and ReLU activations, and explain what each one does to $z$.
- Process a batch of inputs with one matrix–vector product instead of a loop.
- Read a neuron's decision boundary, and explain why no single neuron can compute XOR.

## Dependencies

Tier 1: NumPy and Matplotlib only. Nothing here uses a neural-network library.

### How to work through this notebook

Run the cells from the top. At each **✍️ Your turn**, type the function, then run the `practice` cell under it. It compares your function with a reference on test inputs. Once yours matches, the rest of the notebook runs on **your** code; until then it quietly uses the reference, so you can always keep going.

## Setup

Run this cell first. It loads NumPy and Matplotlib and defines `practice`, the checker used after every ✍️ cell.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import matplotlib.pyplot as plt

## 1 · Inputs, weights and a bias

A neuron receives a vector of inputs $x$. It multiplies each input by a weight, adds the results, and adds a bias:

$$z = w \cdot x + b = \sum_i w_i x_i + b.$$

The weights say how much each input matters and in which direction; the bias shifts the result, so the neuron can respond even when every input is zero. Here is the arithmetic for two inputs:

In [ ]:
x = np.array([1.0, 2.0])
w = np.array([0.5, -1.0])
b = 0.1
print("by hand:", 0.5 * 1.0 + (-1.0) * 2.0 + 0.1)
print("np.dot :", np.dot(w, x) + b)

### ✍️ Your turn: `weighted_sum`

Return $z = w \cdot x + b$ as a float. `x` and `w` have shape `(n_inputs,)`; `b` is a number.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def weighted_sum_reference(x, w, b):
    return float(np.dot(w, x) + b)

In [ ]:
def weighted_sum(x, w, b):
    """z = w · x + b for one input vector x (n_inputs,) and weights w (n_inputs,)."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
weighted_sum = practice("weighted_sum", weighted_sum, weighted_sum_reference, cases=[
    (np.array([1.0, 2.0]), np.array([0.5, -1.0]), 0.1),
    (np.array([0.0, 0.0, 0.0]), np.array([3.0, -2.0, 1.0]), -0.5),
    (np.array([2.0, -1.0, 4.0]), np.array([0.25, 0.5, -0.125]), 0.0),
])

## 2 · Activation functions

On its own, $z$ is a straight-line function of the inputs. The activation $a = f(z)$ bends it. Two common choices:

- **Sigmoid**, $\sigma(z) = \dfrac{1}{1 + e^{-z}}$, squashes any $z$ into $(0, 1)$, which reads naturally as a probability.
- **ReLU**, $\max(0, z)$, passes positive values through and sets negative ones to zero. It is cheap and keeps gradients alive in deep networks.

For very negative $z$, `np.exp(-z)` overflows. A stable sigmoid uses `np.logaddexp(0, -z)`, which computes $\log(1 + e^{-z})$ without overflow: $\sigma(z) = e^{-\log(1+e^{-z})}$.

### ✍️ Your turn: `sigmoid`

Return the sigmoid of every element of `Z` (any shape), stable for large positive and negative values. Hint: `np.exp(-np.logaddexp(0.0, -Z))`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def sigmoid_reference(Z):
    return np.exp(-np.logaddexp(0.0, -Z))

In [ ]:
def sigmoid(Z):
    """Element-wise 1 / (1 + exp(-Z)), without overflow."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
sigmoid = practice("sigmoid", sigmoid, sigmoid_reference, cases=[
    (np.array([0.0, 2.0, -2.0]),),
    (np.array([[1000.0, -1000.0], [0.5, -0.5]]),),
])

### ✍️ Your turn: `relu`

Return `max(0, z)` for every element of `Z` (any shape).

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def relu_reference(Z):
    return np.maximum(0.0, Z)

In [ ]:
def relu(Z):
    """Element-wise max(0, Z)."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
relu = practice("relu", relu, relu_reference, cases=[
    (np.array([-2.0, 0.0, 3.5]),),
    (np.array([[1.0, -1.0], [-0.1, 0.1]]),),
])

In [ ]:
z = np.linspace(-6, 6, 200)
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].plot(z, sigmoid(z)); axes[0].set_title("sigmoid"); axes[0].axhline(0.5, ls=":", c="grey")
axes[1].plot(z, relu(z)); axes[1].set_title("ReLU")
for ax in axes:
    ax.set_xlabel("z")
plt.tight_layout()
plt.show()

## 3 · A whole neuron

Put the two steps together: the neuron's output is $a = f(w \cdot x + b)$.

### ✍️ Your turn: `neuron`

Return `activation(weighted_sum(x, w, b))`. `activation` is a function such as `sigmoid` or `relu`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def neuron_reference(x, w, b, activation):
    return activation(weighted_sum_reference(x, w, b))

In [ ]:
def neuron(x, w, b, activation):
    """One neuron's output for one input vector."""
    # ✍️ Your code here (use your weighted_sum)
    raise NotImplementedError

In [ ]:
neuron = practice("neuron", neuron, neuron_reference, cases=[
    (np.array([1.0, 2.0]), np.array([0.5, -1.0]), 0.1, sigmoid_reference),
    (np.array([1.0, 2.0]), np.array([0.5, -1.0]), 0.1, relu_reference),
    (np.array([3.0, 1.0]), np.array([1.0, 1.0]), -2.0, relu_reference),
])

## 4 · A batch at once

Real training feeds many examples at a time. Stack them as the rows of a matrix $X$ with shape `(batch, n_inputs)`; then all the weighted sums are one product, $z = X w + b$, with shape `(batch,)`. NumPy runs that product in compiled code, far faster than a Python loop.

### ✍️ Your turn: `neuron_batch`

Return the neuron's output for every row of `X` (shape `(batch, n_inputs)`) as an array of shape `(batch,)`. No loops: use `X @ w + b`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def neuron_batch_reference(X, w, b, activation):
    return activation(X @ w + b)

In [ ]:
def neuron_batch(X, w, b, activation):
    """The neuron applied to every row of X: shape (batch,)."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
X_demo = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
neuron_batch = practice("neuron_batch", neuron_batch, neuron_batch_reference, cases=[
    (X_demo, np.array([1.0, 1.0]), -1.5, sigmoid_reference),
    (X_demo, np.array([2.0, -1.0]), 0.25, relu_reference),
])

## 5 · Logic gates by hand

With a large weight, the sigmoid acts almost like a switch. Choose weights and a bias so that $z$ is positive exactly when the gate should say 1:

- **AND**: $w = (10, 10)$, $b = -15$. Only $(1, 1)$ gives $z > 0$.
- **OR**: $w = (10, 10)$, $b = -5$. Any single 1 gives $z > 0$.

The bias is the threshold: moving it from $-15$ to $-5$ turns AND into OR.

In [ ]:
inputs = X_demo
for name, w_gate, b_gate in [("AND", np.array([10.0, 10.0]), -15.0), ("OR", np.array([10.0, 10.0]), -5.0)]:
    out = neuron_batch(inputs, w_gate, b_gate, sigmoid)
    print(name, [f"{a:.3f}" for a in out], "->", (out > 0.5).astype(int).tolist())

### 🧪 Try it

Change the bias in the cell below and re-run it. Which values of `b` give AND, which give OR, and which give neither?

In [ ]:
b_try = -12.0  # try -18, -12, -8, -2, 2
print((neuron_batch(X_demo, np.array([10.0, 10.0]), b_try, sigmoid) > 0.5).astype(int).tolist())

## 6 · What a neuron can draw: a straight line

The neuron says 1 where $w \cdot x + b > 0$. The boundary $w \cdot x + b = 0$ is a straight line in two dimensions (a flat plane in more). The plot colours each point of a grid by the OR neuron's output.

In [ ]:
grid = np.linspace(-0.5, 1.5, 120)
gx, gy = np.meshgrid(grid, grid)
points = np.column_stack([gx.ravel(), gy.ravel()])
out = neuron_batch(points, np.array([10.0, 10.0]), -5.0, sigmoid).reshape(gx.shape)
plt.figure(figsize=(4, 4))
plt.contourf(gx, gy, out, levels=20, cmap="RdBu_r", alpha=0.7)
plt.contour(gx, gy, out, levels=[0.5], colors="k")
plt.scatter(X_demo[:, 0], X_demo[:, 1], c=[0, 1, 1, 1], cmap="RdBu_r", edgecolors="k", s=80)
plt.title("OR neuron: the boundary is a line")
plt.show()

## 7 · XOR: the problem one neuron cannot solve

XOR outputs 1 when exactly one input is 1. Its 1s sit on one diagonal of the square and its 0s on the other, so no single straight line separates them. The cell below tries a grid of 9,261 weight-and-bias combinations and keeps the best accuracy: it never gets past 3 out of 4.

In [ ]:
y_xor = np.array([0, 1, 1, 0])
values = np.linspace(-5, 5, 21)
best = 0.0
for w1 in values:
    for w2 in values:
        for bias in values:
            guess = (neuron_batch(X_demo, np.array([w1, w2]), bias, sigmoid) > 0.5).astype(int)
            best = max(best, float(np.mean(guess == y_xor)))
print("best accuracy any single neuron reached on XOR:", best)

## 8 · Put it together

In [ ]:
and_out = (neuron_batch(X_demo, np.array([10.0, 10.0]), -15.0, sigmoid) > 0.5).astype(int)
or_out = (neuron_batch(X_demo, np.array([10.0, 10.0]), -5.0, sigmoid) > 0.5).astype(int)
assert and_out.tolist() == [0, 0, 0, 1], and_out
assert or_out.tolist() == [0, 1, 1, 1], or_out
assert best <= 0.75, "a single neuron should not solve XOR"
assert np.allclose(neuron_batch(X_demo, np.array([0.3, -0.2]), 0.1, relu),
                   [neuron(row, np.array([0.3, -0.2]), 0.1, relu) for row in X_demo])
print("✅ Your neuron computes AND and OR, works on batches, and cannot do XOR alone.")

## Exercises

1. **NAND.** Find weights and a bias that make the neuron compute NOT-AND (1 except for input $(1, 1)$). Check them with `neuron_batch`.
2. **A third activation.** Write `tanh_activation(Z)` with `np.tanh` and plot it next to the sigmoid. How are the two related? (Hint: compare $\tanh(z)$ with $2\sigma(2z) - 1$.)
3. **Three inputs.** With three inputs, what shape is the boundary $w \cdot x + b = 0$? Describe it in one sentence.
4. **Open question.** XOR needs a curved (or bent) boundary. How could *two* neurons, whose outputs feed a third, produce one? Notebook 03 answers this.

## Common errors and debugging

- **`ValueError: shapes not aligned`**: `X @ w` needs `X` with shape `(batch, n_inputs)` and `w` with `(n_inputs,)`. Print `X.shape` and `w.shape`.
- **`RuntimeWarning: overflow in exp`**: you wrote `1 / (1 + np.exp(-Z))`; use the `logaddexp` form.
- **The check says ❌ but the numbers look right**: compare shapes too. A `(batch, 1)` result is not the same as `(batch,)`.

## Summary

A neuron computes a weighted sum plus a bias and passes it through an activation. With a sigmoid it acts like a soft switch, and its decision boundary is a straight line. That is enough for AND and OR, but not for XOR. Combining neurons into layers fixes that.

## Next

**03 · Build a neural network**: stack neurons into layers, initialise their weights, and solve XOR with two hidden units.